In [2]:
# ============================================================
# OHRC PREPROCESSING - SETUP
# ============================================================

# Local Jupyter / Windows version
# ============================================================

import os
import re
import glob
import math
import json
import zipfile
import tempfile
import shutil
import warnings

from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, List, Dict, Tuple
from datetime import datetime, timezone
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

from PIL import Image

from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# OHRC PROJECT PATHS
# ============================================================

PROJECT_ROOT = r"D:\ISRO"

# Raw OHRC products
DATA_ROOT = os.path.join(PROJECT_ROOT, "OHRC_raws")

# Processed OHRC tiles
OUT_ROOT = os.path.join(PROJECT_ROOT, "OHRC_tiles")

# Create output directory if it does not exist
os.makedirs(OUT_ROOT, exist_ok=True)


# ============================================================
# OHRC TEST SCENE
# ============================================================

SCENE_ID = "ch2_ohr_ncp_20260331T1105235288_d_img_d18"

SAMPLE_ROOT = os.path.join(
    DATA_ROOT,
    SCENE_ID
)


# ============================================================
# TEST FILE PATHS
# ============================================================

# Calibrated image
IMG_PATH = os.path.join(
    SAMPLE_ROOT,
    "data",
    "calibrated",
    "20260331",
    f"{SCENE_ID}.img"
)

# Image PDS4 XML
IMG_XML = os.path.join(
    SAMPLE_ROOT,
    "data",
    "calibrated",
    "20260331",
    f"{SCENE_ID}.xml"
)

# Geometry CSV
GRD_CSV = os.path.join(
    SAMPLE_ROOT,
    "geometry",
    "calibrated",
    "20260331",
    "ch2_ohr_ncp_20260331T1105235288_g_grd_d18.csv"
)

# Geometry XML
GRD_XML = os.path.join(
    SAMPLE_ROOT,
    "geometry",
    "calibrated",
    "20260331",
    "ch2_ohr_ncp_20260331T1105235288_g_grd_d18.xml"
)

# Browse image
BRW_PNG = os.path.join(
    SAMPLE_ROOT,
    "browse",
    "calibrated",
    "20260331",
    "ch2_ohr_ncp_20260331T1105235288_b_brw_d18.png"
)

# Attitude/orbit ancillary file
OATH = os.path.join(
    SAMPLE_ROOT,
    "miscellaneous",
    "calibrated",
    "20260331",
    f"{SCENE_ID}.oath"
)


# ============================================================
# TEST MODE
# ============================================================

TEST_MODE = True

# Final dataset dates
START_DATE = "2026-06-01"
END_DATE = "2026-08-31"


# ============================================================
# PREPROCESSING SETTINGS
# ============================================================

ENCODER_SIZE = 256

TARGET_GROUND_FOOTPRINT_M = 512.0

JPEG_QUALITY = 95

STRIDE_FRACTION = 0.5

MIN_VALID_FRACTION = 0.95

MAX_SATURATION_FRACTION = 0.25


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 70)
print("OHRC PREPROCESSING TEST CONFIGURATION")
print("=" * 70)

print("Project root :", PROJECT_ROOT)
print("Data root    :", DATA_ROOT)
print("Output root  :", OUT_ROOT)
print("Sample scene :", SAMPLE_ROOT)

print("\nTest mode    :", TEST_MODE)
print("Scene ID     :", SCENE_ID)

print("\nImage file   :", IMG_PATH)
print("Image XML    :", IMG_XML)
print("Geometry CSV :", GRD_CSV)
print("Geometry XML :", GRD_XML)
print("Browse PNG   :", BRW_PNG)
print("OATH file    :", OATH)


# ============================================================
# CHECK DIRECTORIES
# ============================================================

print("\n" + "=" * 70)
print("DIRECTORY CHECK")
print("=" * 70)

print("Project root exists :", os.path.exists(PROJECT_ROOT))
print("Data root exists    :", os.path.exists(DATA_ROOT))
print("Sample scene exists :", os.path.exists(SAMPLE_ROOT))
print("Output root exists  :", os.path.exists(OUT_ROOT))


# ============================================================
# CHECK REQUIRED SAMPLE FILES
# ============================================================

print("\n" + "=" * 70)
print("SAMPLE FILE CHECK")
print("=" * 70)

required_files = {
    "img"   : IMG_PATH,
    "img_xml": IMG_XML,
    "grd_csv": GRD_CSV,
    "grd_xml": GRD_XML,
    "brw_png": BRW_PNG,
    "oath"   : OATH,
}

all_found = True

for name, path in required_files.items():

    exists = os.path.exists(path)

    print(f"{name:8s}: {exists}")

    if exists:
        print(f"          {path}")
    else:
        print(f"          MISSING: {path}")
        all_found = False


if all_found:
    print("\n✅ ALL REQUIRED OHRC SAMPLE FILES FOUND")
else:
    print("\n❌ SOME REQUIRED OHRC FILES ARE MISSING")


# ============================================================
# IMAGE FILE SIZE
# ============================================================

if os.path.exists(IMG_PATH):

    size_mb = os.path.getsize(IMG_PATH) / (1024 * 1024)

    print("\nOHRC calibrated image size:",
          round(size_mb, 2), "MB")


print("=" * 70)

OHRC PREPROCESSING TEST CONFIGURATION
Project root : D:\ISRO
Data root    : D:\ISRO\OHRC_raws
Output root  : D:\ISRO\OHRC_tiles
Sample scene : D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18

Test mode    : True
Scene ID     : ch2_ohr_ncp_20260331T1105235288_d_img_d18

Image file   : D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18\data\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d_img_d18.img
Image XML    : D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18\data\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d_img_d18.xml
Geometry CSV : D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18\geometry\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_g_grd_d18.csv
Geometry XML : D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18\geometry\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_g_grd_d18.xml
Browse PNG   : D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18\browse\calibrated\20260331\ch2_ohr_ncp_20260331T110

In [3]:
# ============================================================
# OHRC SCENE DISCOVERY AND DATE PARSING
# ============================================================

OHRC_NAME_RE = re.compile(
    r"ch2_ohr_(?P<mode>[a-z0-9]+)_(?P<stamp>\d{8}T\d{10})_d_img_[a-z0-9]+",
    re.IGNORECASE,
)


def parse_acquisition_from_name(name: str) -> Optional[datetime]:
    """
    Extract acquisition UTC time from an OHRC scene/file name.

    Example:
    ch2_ohr_ncp_20260331T1105235288_d_img_d18
    """

    m = OHRC_NAME_RE.search(os.path.basename(name))

    if not m:
        return None

    # Stamp format:
    # YYYYMMDDTHHMMSSssss
    # where ssss represents fractional seconds.

    stamp = m.group("stamp")

    base = datetime.strptime(
        stamp[:15],
        "%Y%m%dT%H%M%S"
    )

    frac4 = int(stamp[15:19])

    return base.replace(
        microsecond=frac4 * 100
    )


def is_calibrated_name(name: str) -> bool:
    """
    OHRC calibrated image products use the NCP mode.
    """

    m = OHRC_NAME_RE.search(
        os.path.basename(name)
    )

    return bool(
        m and
        "ncp" in m.group("mode").lower()
    )


# ============================================================
# FIND EXTRACTED OHRC SCENE DIRECTORIES
# ============================================================

def find_ohrc_scene_groups(
    data_root=DATA_ROOT,
    calibrated_only=True
) -> pd.DataFrame:

    rows = []

    # Each immediate directory under OHRC_raws
    # represents one extracted OHRC product/scene.
    for scene_dir in Path(data_root).iterdir():

        if not scene_dir.is_dir():
            continue

        scene_name = scene_dir.name

        acquisition_dt = parse_acquisition_from_name(
            scene_name
        )

        if acquisition_dt is None:
            continue

        if calibrated_only and not is_calibrated_name(
            scene_name
        ):
            continue

        # Find the calibrated image recursively.
        img_candidates = list(
            scene_dir.rglob("*.img")
        )

        # Find geometry CSV recursively.
        grd_candidates = list(
            scene_dir.rglob("*_g_grd_*.csv")
        )

        # Prefer the image whose filename matches
        # the scene directory name.
        matching_img = [
            p for p in img_candidates
            if p.stem.lower() == scene_name.lower()
        ]

        if matching_img:
            img_path = matching_img[0]
        elif img_candidates:
            img_path = img_candidates[0]
        else:
            img_path = None

        grd_path = (
            grd_candidates[0]
            if grd_candidates
            else None
        )

        rows.append({
            "scene_id": scene_name,
            "scene_root": str(scene_dir),
            "img_path": str(img_path) if img_path else "",
            "grd_csv": str(grd_path) if grd_path else "",
            "acquisition_utc": acquisition_dt.isoformat(),
            "calibrated": is_calibrated_name(scene_name),
        })

    if not rows:
        return pd.DataFrame(
            columns=[
                "scene_id",
                "scene_root",
                "img_path",
                "grd_csv",
                "acquisition_utc",
                "calibrated",
            ]
        )

    return pd.DataFrame(rows).sort_values(
        "acquisition_utc"
    ).reset_index(drop=True)


# ============================================================
# TEST SCENE DISCOVERY
# ============================================================

selected = find_ohrc_scene_groups()

print("=" * 70)
print("OHRC SCENE DISCOVERY")
print("=" * 70)

print("Data root:", DATA_ROOT)

print("\nScenes found:", len(selected))

if len(selected) > 0:

    display(selected)

else:

    print("\n❌ No calibrated OHRC scenes found.")

print("=" * 70)

OHRC SCENE DISCOVERY
Data root: D:\ISRO\OHRC_raws

Scenes found: 1


,scene_id,scene_root,img_path,grd_csv,acquisition_utc,calibrated
0,ch2_ohr_ncp_20260331T1105235288_d_img_d18,D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235...,D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235...,D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235...,2026-03-31T11:05:23.528800,True


In [4]:
# ============================================================
# INSPECT EXTRACTED OHRC SCENE
# ============================================================

def inspect_ohrc_scene(scene_root: str, max_entries=80) -> List[str]:
    """
    Inspect the files contained inside an extracted OHRC scene.
    Equivalent to the old inspect_zip() function, but works
    with the extracted PDS4 directory structure.
    """

    scene_root = os.path.abspath(scene_root)

    if not os.path.exists(scene_root):
        raise FileNotFoundError(
            f"OHRC scene directory not found:\n{scene_root}"
        )

    print("=" * 70)
    print("OHRC SCENE INSPECTION")
    print("=" * 70)

    print("Scene root:")
    print(scene_root)

    # Find all files recursively
    files = [
        p for p in Path(scene_root).rglob("*")
        if p.is_file()
    ]

    print(f"\nTotal files: {len(files)}")

    print("\nFiles:")
    print("-" * 70)

    for p in files[:max_entries]:
        relative_path = p.relative_to(scene_root)

        size_mb = p.stat().st_size / (1024 * 1024)

        print(
            f"{relative_path}"
            f"    [{size_mb:.2f} MB]"
        )

    if len(files) > max_entries:
        print(
            f"\n... {len(files) - max_entries} more files"
        )

    print("=" * 70)

    return [str(p) for p in files]


# ============================================================
# INSPECT OUR TEST OHRC SCENE
# ============================================================

scene_files = inspect_ohrc_scene(
    SAMPLE_ROOT
)

OHRC SCENE INSPECTION
Scene root:
D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18

Total files: 11

Files:
----------------------------------------------------------------------
miscellaneous\readme.txt    [0.01 MB]
browse\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_b_brw_d18.png    [4.76 MB]
browse\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_b_brw_d18.xml    [0.00 MB]
data\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d_img_d18.img    [1072.22 MB]
data\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d_img_d18.xml    [0.01 MB]
geometry\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_g_grd_d18.csv    [3.65 MB]
geometry\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_g_grd_d18.xml    [0.01 MB]
miscellaneous\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d_img_d18.lbr    [0.16 MB]
miscellaneous\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d_img_d18.oat    [0.39 MB]
miscellaneous\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d

In [5]:
def _local(tag: str) -> str:
    return tag.split("}")[-1]


def _text_by_local(root, name: str, default=None):
    for el in root.iter():
        if _local(el.tag) == name and el.text is not None:
            return el.text.strip()
    return default


def _float_by_local(root, name: str, default=np.nan):
    x = _text_by_local(root, name, None)
    try:
        return float(x)
    except Exception:
        return default


def parse_ohrc_label(xml_path: str) -> dict:
    root = ET.parse(xml_path).getroot()

    axes = {}

    for aa in root.iter():
        if _local(aa.tag) != "Axis_Array":
            continue

        axis_name = _text_by_local(aa, "axis_name", "")
        elements = _text_by_local(aa, "elements", None)

        if axis_name and elements:
            axes[axis_name.lower()] = int(elements)

    file_name = _text_by_local(root, "file_name")

    offset = int(float(_text_by_local(root, "offset", 0)))

    data_type = _text_by_local(
        root,
        "data_type",
        "UnsignedByte"
    )

    start_time = _text_by_local(root, "start_date_time")
    stop_time = _text_by_local(root, "stop_date_time")

    corners = {}

    for k in [
        "upper_left_latitude",
        "upper_left_longitude",
        "upper_right_latitude",
        "upper_right_longitude",
        "lower_left_latitude",
        "lower_left_longitude",
        "lower_right_latitude",
        "lower_right_longitude",
    ]:
        corners[k] = _float_by_local(root, k)

    meta = {
        "file_name": file_name,
        "lines": axes.get("line"),
        "samples": axes.get("sample"),
        "offset": offset,
        "data_type": data_type,
        "start_date_time": start_time,
        "stop_date_time": stop_time,

        "processing_level": _text_by_local(
            root, "processing_level"
        ),

        "imaging_orbit_number": _text_by_local(
            root, "imaging_orbit_number"
        ),

        "tdi_stages": _text_by_local(
            root, "tdi_stages"
        ),

        "bits_selection": _text_by_local(
            root, "bits_selection"
        ),

        "line_exposure_duration_ms": _float_by_local(
            root, "line_exposure_duration"
        ),

        "spacecraft_altitude_km": _float_by_local(
            root, "spacecraft_altitude"
        ),

        "pixel_resolution_m": _float_by_local(
            root,
            "pixel_resolution",
            default=0.25
        ),

        "sun_azimuth": _float_by_local(
            root, "sun_azimuth"
        ),

        "sun_elevation": _float_by_local(
            root, "sun_elevation"
        ),

        "solar_incidence": _float_by_local(
            root, "solar_incidence"
        ),

        "orbit_limb_direction": _text_by_local(
            root, "orbit_limb_direction"
        ),

        "spacecraft_yaw_direction": _text_by_local(
            root, "spacecraft_yaw_direction"
        ),

        "reference_data_used": _text_by_local(
            root, "reference_data_used"
        ),

        **corners,
    }

    if not meta["lines"] or not meta["samples"]:
        raise ValueError(
            "Could not read Line/Sample dimensions from PDS4 label"
        )

    if str(data_type).lower() not in {
        "unsignedbyte",
        "unsigned byte",
        "uint8",
    }:
        print(
            f"[WARN] Expected 8-bit UnsignedByte, "
            f"label reports {data_type}"
        )

    return meta

In [6]:
def open_ohrc_memmap(img_path: str, meta: dict) -> np.memmap:
    """
    Open the OHRC .img file as a memory-mapped 8-bit raster.

    The complete image is NOT loaded into RAM.
    """

    dtype = np.uint8  # OHRC PDS4 SIS: UnsignedByte

    return np.memmap(
        img_path,
        dtype=dtype,
        mode="r",
        offset=int(meta.get("offset", 0)),
        shape=(
            int(meta["lines"]),
            int(meta["samples"])
        ),
        order="C"
    )


def orientation_ops(meta: dict) -> Tuple[bool, bool]:
    """
    Return (reverse_lines, reverse_samples)
    according to the OHRC archive orientation rules.

    Descending + False -> none
    Descending + True  -> reverse samples
    Ascending  + False -> reverse lines
    Ascending  + True  -> reverse both
    """

    limb = str(
        meta.get("orbit_limb_direction", "")
    ).strip().lower()

    yaw_txt = str(
        meta.get("spacecraft_yaw_direction", "")
    ).strip().lower()

    yaw_true = yaw_txt in {
        "true",
        "1",
        "yes",
        "reverse"
    }

    reverse_lines = limb.startswith("asc")
    reverse_samples = yaw_true

    return reverse_lines, reverse_samples


def crop_oriented(
    mm: np.ndarray,
    r0: int,
    c0: int,
    h: int,
    w: int,
    meta: dict
) -> np.ndarray:
    """
    Read an oriented crop without loading the full OHRC scene.
    """

    H, W = mm.shape

    rev_r, rev_c = orientation_ops(meta)

    # Map oriented crop coordinates back
    # into the stored raster coordinates.

    sr0 = H - (r0 + h) if rev_r else r0
    sc0 = W - (c0 + w) if rev_c else c0

    arr = np.asarray(
        mm[
            sr0:sr0 + h,
            sc0:sc0 + w
        ]
    )

    # Restore the desired orientation.
    if rev_r:
        arr = arr[::-1, :]

    if rev_c:
        arr = arr[:, ::-1]

    # Return an independent NumPy array.
    return arr.copy()

In [7]:
KNOWN_MAX_SR = {
    64: 3.87,
    256: 0.8
}


def tdi_number(meta: dict) -> Optional[int]:
    m = re.search(
        r"(\d+)",
        str(meta.get("tdi_stages", ""))
    )

    return int(m.group(1)) if m else None


def counts_to_radiance(
    dn: np.ndarray,
    meta: dict
) -> np.ndarray:

    tdi = tdi_number(meta)

    if tdi not in KNOWN_MAX_SR:
        raise ValueError(
            f"No Max_SR supplied in the OHRC SIS for TDI{tdi}; "
            "do not guess it."
        )

    # SIS:
    # radiance = ((Max_SR - Min_SR) * gray_value) / 2^8
    # Min_SR = 0

    return (
        KNOWN_MAX_SR[tdi]
        * dn.astype(np.float32)
        / 256.0
    ).astype(np.float32)


def robust_unit_range(
    arr: np.ndarray,
    lo_pct=1.0,
    hi_pct=99.0
):
    x = arr.astype(np.float32)

    valid = x[np.isfinite(x)]

    lo, hi = np.percentile(
        valid,
        [lo_pct, hi_pct]
    )

    y = np.clip(
        (x - lo) / max(float(hi - lo), 1e-6),
        0,
        1
    )

    return (
        y.astype(np.float32),
        float(lo),
        float(hi)
    )


def saturation_fraction(
    dn: np.ndarray
) -> float:

    return float(
        np.mean(dn >= 255)
    )

In [8]:
def illumination_invariant_weber(
    structural_unit: np.ndarray,
    sigma_px: float = 25.0,
    eps: float = 1e-3
) -> np.ndarray:

    # Estimate the local illumination/background.
    local_mean = gaussian_filter(
        structural_unit.astype(np.float32),
        sigma=sigma_px
    )

    # Weber contrast:
    # measures local change relative to local illumination.
    w = (
        structural_unit - local_mean
    ) / (
        local_mean + eps
    )

    # Robustly normalize to [0, 1].
    lo, hi = np.percentile(
        w[np.isfinite(w)],
        [1, 99]
    )

    return np.clip(
        (w - lo) / max(float(hi - lo), 1e-6),
        0,
        1
    ).astype(np.float32)

In [9]:
class OHRCGeoGrid:

    def __init__(self, csv_path: str):

        df = pd.read_csv(csv_path)

        # Normalize column names so that small differences
        # in formatting/capitalization do not matter.
        norm = {
            c: re.sub(
                r"[^a-z0-9]",
                "",
                c.lower()
            )
            for c in df.columns
        }

        def find_col(keys):
            # First try exact normalized matches.
            for c, n in norm.items():
                if n in keys:
                    return c

            # Then try partial matches.
            for c, n in norm.items():
                if any(k in n for k in keys):
                    return c

            return None

        self.lon_col = find_col({
            "longitude",
            "lon"
        })

        self.lat_col = find_col({
            "latitude",
            "lat"
        })

        self.pix_col = find_col({
            "pixel",
            "sample",
            "samples"
        })

        self.scan_col = find_col({
            "scan",
            "line",
            "lines"
        })

        if not all([
            self.lon_col,
            self.lat_col,
            self.pix_col,
            self.scan_col
        ]):
            raise ValueError(
                "Could not identify geometry columns in "
                f"{list(df.columns)}"
            )

        self.points = df[
            [self.pix_col, self.scan_col]
        ].astype(float).to_numpy()

        self.lons = df[
            self.lon_col
        ].astype(float).to_numpy()

        self.lats = df[
            self.lat_col
        ].astype(float).to_numpy()

    def at(
        self,
        pixel: float,
        scan: float
    ) -> Tuple[float, float]:

        q = np.array(
            [[pixel, scan]],
            dtype=float
        )

        # Linear interpolation.
        lon = griddata(
            self.points,
            self.lons,
            q,
            method="linear"
        )[0]

        lat = griddata(
            self.points,
            self.lats,
            q,
            method="linear"
        )[0]

        # If the requested point falls outside the
        # interpolation region, use nearest-neighbour.
        if np.isnan(lon) or np.isnan(lat):

            lon = griddata(
                self.points,
                self.lons,
                q,
                method="nearest"
            )[0]

            lat = griddata(
                self.points,
                self.lats,
                q,
                method="nearest"
            )[0]

        return float(lat), float(lon)

In [ ]:
@dataclass
class OHRCPatchMeta:
    source_scene_id: str
    source_scene_root: str
    acquisition_utc: Optional[str]
    tile_row: int
    tile_col: int
    native_r0: int
    native_c0: int
    native_size_px: int
    encoder_size_px: int
    ground_footprint_m: float
    resolution_m_per_px: float
    lat: float
    lon: float
    sun_azimuth: float
    sun_elevation: float
    solar_incidence: float
    tdi_stages: str
    bits_selection: str
    orbit_limb_direction: str
    spacecraft_yaw_direction: str
    saturation_fraction: float
    structural_jpeg: str
    illumination_jpeg: str


def save_gray_jpeg(
    unit_img: np.ndarray,
    path: str,
    size=ENCODER_SIZE,
    quality=JPEG_QUALITY
):
    u8 = (
        np.clip(unit_img, 0, 1) * 255.0 + 0.5
    ).astype(np.uint8)

    im = Image.fromarray(
        u8,
        mode="L"
    )

    if im.size != (size, size):
        im = im.resize(
            (size, size),
            resample=Image.Resampling.LANCZOS
        )

    im.save(
        path,
        format="JPEG",
        quality=quality,
        subsampling=0,
        optimize=True
    )


def tile_ohrc_scene(
    mm: np.ndarray,
    meta: dict,
    geo: Optional[OHRCGeoGrid],
    scene_id: str,
    source_scene_root: str,
    out_dir: str,
    target_ground_m=TARGET_GROUND_FOOTPRINT_M,
    stride_fraction=STRIDE_FRACTION
) -> List[dict]:

    # ---------------------------------------------------------
    # Ground sampling distance in metres/pixel.
    # ---------------------------------------------------------

    gsd = float(
        meta.get(
            "pixel_resolution_m",
            0.25
        )
    )

    if not np.isfinite(gsd) or gsd <= 0:
        gsd = 0.25

    # ---------------------------------------------------------
    # Convert desired ground footprint to native pixels.
    # ---------------------------------------------------------

    native = max(
        256,
        int(round(target_ground_m / gsd))
    )

    # Never request a tile larger than the image.
    native = min(
        native,
        mm.shape[0],
        mm.shape[1]
    )

    # ---------------------------------------------------------
    # 50% overlap by default.
    # ---------------------------------------------------------

    stride = max(
        1,
        int(round(native * stride_fraction))
    )

    H, W = mm.shape

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    # =========================================================
    # CALCULATE ALL TILE POSITIONS
    # =========================================================

    row_positions = list(
        range(
            0,
            max(H - native + 1, 1),
            stride
        )
    )

    col_positions = list(
        range(
            0,
            max(W - native + 1, 1),
            stride
        )
    )

    total_tiles = (
        len(row_positions) *
        len(col_positions)
    )

    print(
        f"\nNative tile size      : "
        f"{native} x {native} px"
    )

    print(
        f"Ground footprint      : "
        f"{native * gsd:.2f} m"
    )

    print(
        f"Stride                : "
        f"{stride} px"
    )

    print(
        f"Image size            : "
        f"{H} x {W} px"
    )

    print(
        f"Total tile positions  : "
        f"{total_tiles}"
    )

    print("-" * 70)

    # =========================================================
    # PROGRESS COUNTERS
    # =========================================================

    processed_tiles = 0
    completed_tiles = 0
    skipped_tiles = 0

    rows = []

    tr = 0

    # =========================================================
    # TILE LOOP
    # =========================================================

    for r0 in row_positions:

        tc = 0

        for c0 in col_positions:

            # -------------------------------------------------
            # Read only the required oriented crop.
            # -------------------------------------------------

            dn = crop_oriented(
                mm,
                r0,
                c0,
                native,
                native,
                meta
            )

            processed_tiles += 1

            # -------------------------------------------------
            # Reject invalid tile shapes.
            # -------------------------------------------------

            if dn.shape != (native, native):

                skipped_tiles += 1

                remaining_tiles = (
                    total_tiles -
                    processed_tiles
                )

                print(
                    f"\rTiles processed: "
                    f"{processed_tiles}/{total_tiles} | "
                    f"Completed: {completed_tiles} | "
                    f"Skipped: {skipped_tiles} | "
                    f"Remaining: {remaining_tiles}",
                    end="",
                    flush=True
                )

                tc += 1
                continue

            # -------------------------------------------------
            # Reject heavily saturated tiles.
            # -------------------------------------------------

            sat = saturation_fraction(dn)

            if sat > MAX_SATURATION_FRACTION:

                skipped_tiles += 1

                remaining_tiles = (
                    total_tiles -
                    processed_tiles
                )

                print(
                    f"\rTiles processed: "
                    f"{processed_tiles}/{total_tiles} | "
                    f"Completed: {completed_tiles} | "
                    f"Skipped: {skipped_tiles} | "
                    f"Remaining: {remaining_tiles}",
                    end="",
                    flush=True
                )

                tc += 1
                continue

            # -------------------------------------------------
            # Structural channel.
            # -------------------------------------------------

            structural, _, _ = robust_unit_range(
                dn
            )

            # -------------------------------------------------
            # Illumination-invariant channel.
            # -------------------------------------------------

            illum = illumination_invariant_weber(
                structural
            )

            # -------------------------------------------------
            # Default to NaN if geometry is unavailable.
            # -------------------------------------------------

            lat = lon = float("nan")

            if geo is not None:

                try:

                    # Tile centre in oriented coordinates.
                    cr = r0 + native / 2
                    cc = c0 + native / 2

                    rev_r, rev_c = orientation_ops(
                        meta
                    )

                    # Convert oriented coordinates back
                    # to stored raster coordinates.
                    scan = (
                        H - 1 - cr
                        if rev_r
                        else cr
                    )

                    pixel = (
                        W - 1 - cc
                        if rev_c
                        else cc
                    )

                    lat, lon = geo.at(
                        pixel,
                        scan
                    )

                except Exception:
                    pass

            # -------------------------------------------------
            # Generate filenames.
            # -------------------------------------------------

            stem = (
                f"{scene_id}_"
                f"r{tr:05d}_"
                f"c{tc:03d}"
            )

            p_struct = os.path.join(
                out_dir,
                stem + "_structural.jpg"
            )

            p_illum = os.path.join(
                out_dir,
                stem + "_illum.jpg"
            )

            # -------------------------------------------------
            # Save structural image.
            # -------------------------------------------------

            save_gray_jpeg(
                structural,
                p_struct
            )

            # -------------------------------------------------
            # Save illumination image.
            # -------------------------------------------------

            save_gray_jpeg(
                illum,
                p_illum
            )

            # -------------------------------------------------
            # Save metadata.
            # -------------------------------------------------

            rows.append(
                asdict(
                    OHRCPatchMeta(
                        source_scene_id=scene_id,
                        source_scene_root=os.path.abspath(
                            source_scene_root
                        ),
                        acquisition_utc=meta.get(
                            "start_date_time"
                        ),
                        tile_row=tr,
                        tile_col=tc,
                        native_r0=r0,
                        native_c0=c0,
                        native_size_px=native,
                        encoder_size_px=ENCODER_SIZE,
                        ground_footprint_m=native * gsd,
                        resolution_m_per_px=gsd,
                        lat=lat,
                        lon=lon,
                        sun_azimuth=float(
                            meta.get(
                                "sun_azimuth",
                                np.nan
                            )
                        ),
                        sun_elevation=float(
                            meta.get(
                                "sun_elevation",
                                np.nan
                            )
                        ),
                        solar_incidence=float(
                            meta.get(
                                "solar_incidence",
                                np.nan
                            )
                        ),
                        tdi_stages=str(
                            meta.get(
                                "tdi_stages",
                                ""
                            )
                        ),
                        bits_selection=str(
                            meta.get(
                                "bits_selection",
                                ""
                            )
                        ),
                        orbit_limb_direction=str(
                            meta.get(
                                "orbit_limb_direction",
                                ""
                            )
                        ),
                        spacecraft_yaw_direction=str(
                            meta.get(
                                "spacecraft_yaw_direction",
                                ""
                            )
                        ),
                        saturation_fraction=sat,
                        structural_jpeg=p_struct,
                        illumination_jpeg=p_illum,
                    )
                )
            )

            # -------------------------------------------------
            # Successful tile.
            # -------------------------------------------------

            completed_tiles += 1

            remaining_tiles = (
                total_tiles -
                processed_tiles
            )

            # -------------------------------------------------
            # LIVE PROGRESS
            # -------------------------------------------------

            print(
                f"\rTiles processed: "
                f"{processed_tiles}/{total_tiles} | "
                f"Completed: {completed_tiles} | "
                f"Skipped: {skipped_tiles} | "
                f"Remaining: {remaining_tiles}",
                end="",
                flush=True
            )

            tc += 1

        tr += 1

    # =========================================================
    # FINISHED
    # =========================================================

    print()

    print("-" * 70)

    print(
        "Tile generation complete."
    )

    print(
        f"Total positions : {total_tiles}"
    )

    print(
        f"Completed       : {completed_tiles}"
    )

    print(
        f"Skipped         : {skipped_tiles}"
    )

    print(
        "Remaining       : 0"
    )

    print(
        f"JPEG patch pairs: {len(rows)}"
    )

    print("-" * 70)

    return rows

In [ ]:
def process_ohrc_scene(
    scene_root: str,
    out_root=OUT_ROOT
) -> List[dict]:

    scene_root = os.path.abspath(scene_root)
    scene_id = os.path.basename(
        os.path.normpath(scene_root)
    )

    rows = []

    try:
        # Locate the files inside the already-extracted scene.
        img_candidates = list(
            Path(scene_root).rglob("*.img")
        )

        xml_candidates = list(
            Path(scene_root).rglob("*.xml")
        )

        grd_candidates = list(
            Path(scene_root).rglob("*_g_grd_*.csv")
        )

        # Prefer the XML whose filename matches the scene ID.
        matching_xml = [
            p for p in xml_candidates
            if p.stem.lower() == scene_id.lower()
        ]

        img_path = None
        if img_candidates:
            matching_img = [
                p for p in img_candidates
                if p.stem.lower() == scene_id.lower()
            ]

            img_path = (
                matching_img[0]
                if matching_img
                else img_candidates[0]
            )

        img_xml = (
            matching_xml[0]
            if matching_xml
            else None
        )

        grd_csv = (
            grd_candidates[0]
            if grd_candidates
            else None
        )

        if img_xml is None or img_path is None:
            print(
                f"[SKIP] {scene_id}: "
                "full-resolution image XML/IMG not found"
            )
            return []

        print(f"\nProcessing scene: {scene_id}")
        print(f"IMG: {img_path}")
        print(f"XML: {img_xml}")

        if grd_csv:
            print(f"Geometry: {grd_csv}")
        else:
            print("[WARN] Geometry CSV not found")

        # Parse OHRC metadata.
        meta = parse_ohrc_label(
            str(img_xml)
        )

        # Open the large image using memory mapping.
        mm = open_ohrc_memmap(
            str(img_path),
            meta
        )

        geo = None

        if grd_csv:
            try:
                geo = OHRCGeoGrid(
                    str(grd_csv)
                )
            except Exception as e:
                print(
                    f"[WARN] geometry grid: {e}"
                )

        # Create output directory for this scene.
        scene_out = os.path.join(
            out_root,
            scene_id
        )

        # Generate OHRC tile pairs.
        rows = tile_ohrc_scene(
            mm,
            meta,
            geo,
            scene_id,
            scene_root,
            scene_out
        )

        del mm

        print(
            f"[OK] {scene_id}: "
            f"{len(rows)} JPEG patch pairs"
        )

    except Exception as e:
        print(
            f"[ERROR] {scene_id}: {e!r}"
        )

    return rows


def run_ohrc_batch(
    data_root=DATA_ROOT,
    out_root=OUT_ROOT,
    limit: Optional[int] = None
):

    selected = find_ohrc_scene_groups(
        data_root=data_root,
        calibrated_only=True
    )

    # For the current smoke test, do NOT apply the
    # original June-August filter because our available
    # test scene is from March 31, 2026.
    if limit is not None:
        selected = selected.head(limit)

    print("=" * 70)
    print("OHRC BATCH PROCESSING")
    print("=" * 70)

    print(
        f"Found {len(selected)} calibrated OHRC scene(s)"
    )

    if len(selected) == 0:
        print(
            "\n❌ No calibrated OHRC scenes found."
        )

        return None

    all_rows = []

    for i, row in selected.reset_index(drop=True).iterrows():

        scene_id = row["scene_id"]
        scene_root = row["scene_root"]

        print(
            f"\n--- [{i+1}/{len(selected)}] "
            f"{scene_id} ---"
        )

        all_rows.extend(
            process_ohrc_scene(
                scene_root,
                out_root
            )
        )

    manifest = os.path.join(
        out_root,
        "ohrc_manifest.csv"
    )

    pd.DataFrame(
        all_rows
    ).to_csv(
        manifest,
        index=False
    )

    print(
        f"\nWrote {len(all_rows)} patch rows "
        f"-> {manifest}"
    )

    return manifest


# First smoke-test ONE scene:
# manifest = run_ohrc_batch(limit=1)

# Then test multiple scenes:
# manifest = run_ohrc_batch(limit=5)

# Finally process all discovered calibrated scenes:
# manifest = run_ohrc_batch()

OHRC BATCH PROCESSING
Found 1 calibrated OHRC scene(s)

--- [1/1] ch2_ohr_ncp_20260331T1105235288_d_img_d18 ---

Processing scene: ch2_ohr_ncp_20260331T1105235288_d_img_d18
IMG: D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18\data\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d_img_d18.img
XML: D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18\data\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_d_img_d18.xml
Geometry: D:\ISRO\OHRC_raws\ch2_ohr_ncp_20260331T1105235288_d_img_d18\geometry\calibrated\20260331\ch2_ohr_ncp_20260331T1105235288_g_grd_d18.csv


In [12]:
def show_patch_pair(
    structural_jpeg: str,
    illumination_jpeg: str
):

    a = np.array(
        Image.open(
            structural_jpeg
        ).convert("L")
    )

    b = np.array(
        Image.open(
            illumination_jpeg
        ).convert("L")
    )

    fig, ax = plt.subplots(
        1,
        2,
        figsize=(8, 4)
    )

    ax[0].imshow(
        a,
        cmap="gray"
    )
    ax[0].set_title(
        "OHRC structural JPEG"
    )
    ax[0].axis("off")

    ax[1].imshow(
        b,
        cmap="gray"
    )
    ax[1].set_title(
        "OHRC illumination-invariant JPEG"
    )
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader


class OHRCPatchDataset(Dataset):

    def __init__(self, manifest_csv: str):
        self.df = pd.read_csv(manifest_csv)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Load structural channel.
        s = np.array(
            Image.open(
                row.structural_jpeg
            ).convert("L"),
            dtype=np.float32
        ) / 255.0

        # Load illumination-invariant channel.
        i = np.array(
            Image.open(
                row.illumination_jpeg
            ).convert("L"),
            dtype=np.float32
        ) / 255.0

        # Stack the two grayscale images into
        # a 2-channel tensor: [2, H, W].
        x = torch.from_numpy(
            np.stack(
                [s, i],
                axis=0
            )
        ).float()

        return x, row.to_dict()

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm


class OHRCEncoder(nn.Module):

    def __init__(
        self,
        in_channels=2,
        embed_dim=256,
        proj_dim=128,
        pretrained=True
    ):
        super().__init__()

        # Use ImageNet-pretrained ResNet18 when requested.
        weights = (
            tvm.ResNet18_Weights.IMAGENET1K_V1
            if pretrained
            else None
        )

        backbone = tvm.resnet18(
            weights=weights
        )

        # Original ResNet18 expects 3-channel RGB.
        # OHRC has 2 channels:
        #   channel 1 = structural
        #   channel 2 = illumination-invariant
        old = backbone.conv1

        new = nn.Conv2d(
            in_channels,
            old.out_channels,
            kernel_size=old.kernel_size,
            stride=old.stride,
            padding=old.padding,
            bias=False
        )

        # Adapt the pretrained first layer to 2 channels.
        with torch.no_grad():

            if pretrained:
                w = old.weight.mean(
                    dim=1,
                    keepdim=True
                )

                new.weight.copy_(
                    w.repeat(
                        1,
                        in_channels,
                        1,
                        1
                    )
                    / in_channels
                    * 3.0
                )

        backbone.conv1 = new

        # Remove the original ResNet classification layer.
        self.backbone = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        # 512-D ResNet18 feature → 256-D embedding.
        self.embed = nn.Linear(
            backbone.fc.in_features,
            embed_dim
        )

        # Projection head used for contrastive training.
        self.proj_head = nn.Sequential(
            nn.Linear(
                embed_dim,
                embed_dim
            ),
            nn.ReLU(inplace=True),
            nn.Linear(
                embed_dim,
                proj_dim
            )
        )

    def forward(
        self,
        x,
        project=True
    ):

        # Extract ResNet features.
        feat = self.backbone(
            x
        ).flatten(1)

        # Produce the 256-D sensor embedding.
        emb = self.embed(
            feat
        )

        if not project:
            return emb

        # Produce normalized 128-D contrastive projection.
        proj = F.normalize(
            self.proj_head(emb),
            dim=-1
        )

        return emb, proj


def info_nce_loss(
    proj_ohrc,
    proj_tmc,
    temperature=0.07
):

    # Normalize both sensor projections.
    a = F.normalize(
        proj_ohrc,
        dim=-1
    )

    b = F.normalize(
        proj_tmc,
        dim=-1
    )

    # Pairwise OHRC ↔ TMC similarity matrix.
    logits = (
        a @ b.T
    ) / temperature

    # Correct match is the same batch index.
    labels = torch.arange(
        logits.shape[0],
        device=logits.device
    )

    # Symmetric cross-modal InfoNCE:
    # OHRC → TMC and TMC → OHRC.
    return 0.5 * (
        F.cross_entropy(
            logits,
            labels
        )
        +
        F.cross_entropy(
            logits.T,
            labels
        )
    )


# ---------------------------------------------------------
# Architecture self-test
# ---------------------------------------------------------
# pretrained=False means no ImageNet weights are downloaded.
enc = OHRCEncoder(
    pretrained=False
)

dummy = torch.randn(
    2,
    2,
    ENCODER_SIZE,
    ENCODER_SIZE
)

emb, proj = enc(
    dummy
)

print(
    "embedding",
    emb.shape,
    "projection",
    proj.shape
)

embedding torch.Size([2, 256]) projection torch.Size([2, 128])


In [15]:
def shannon_entropy_u8(
    img_u8: np.ndarray
) -> float:

    hist = np.bincount(
        img_u8.ravel(),
        minlength=256
    ).astype(np.float64)

    p = hist / max(
        hist.sum(),
        1
    )

    p = p[p > 0]

    return float(
        -(p * np.log2(p)).sum()
    )


def low_texture_risk(
    jpeg_path: str,
    entropy_threshold=3.0
) -> bool:

    img = np.array(
        Image.open(
            jpeg_path
        ).convert("L")
    )

    return (
        shannon_entropy_u8(img)
        < entropy_threshold
    )